In [1]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

In [2]:
n_qubits = 8
n_layers = 3

dev = qml.device("lightning.gpu",wires=n_qubits)

In [3]:
@qml.qnode(dev)
def circuit(weights):

    for layer in range(n_layers):
    
        for q in range(n_qubits):
            qml.RY(weights[layer, q],wires=q)
            qml.RY(weights[layer, q],wires=q)
            qml.RY(weights[layer, q],wires=q)
            
        #for q in range(n_qubits-1):
            #qml.CNOT(wires=[q,q + 1])

    return qml.expval(qml.PauliZ(0))

In [4]:
weights = np.random.uniform(-np.pi,np.pi,size=(n_layers,n_qubits),requires_grad=True)
print(weights)
qml.draw_mpl(circuit,style='pennylane')(weights);

[[-2.74362758  1.61694334  0.25466676 -2.57456179 -2.47068065  1.90210992
  -0.66515513 -0.86093574]
 [ 2.88957693  2.51758847 -1.12654592  1.71435379 -2.8335004   2.77747255
  -0.26370416 -2.56535301]
 [ 1.08688652  2.56132755 -2.64478922 -1.08169944 -0.58504768 -1.58838041
  -2.22978761  0.06034098]]


ValueError: PauliZ: wrong number of wires. 7 wires given, 1 expected.

In [ ]:
result = circuit(weights)

print(result)

In [ ]:
gradient = qml.grad(circuit)(weights)
print(gradient)

In [ ]:
#1st test

n_trials = 1000
gradients = []

for trials in range(n_trials):

    weights = np.random.uniform(-np.pi,np.pi,size=(n_layers,n_qubits),requires_grad=True)

    gradient = qml.grad(circuit)(weights)

    gradients.append(gradient)

In [ ]:
gradients = np.array(gradients)
parameter_gradients = gradients[:,0,0]

mean = np.mean(parameter_gradients)
variance = np.var(parameter_gradients)
deviation = np.std(parameter_gradients)

print(parameter_gradients[:10])
print(mean)
print(variance)
print(deviation)

#for n = 4 var = 0.224397860761738, 0.23796733674128637
#n = 6 0.24755993637102627, 0.23575859786200715
#n = 8 0.2441412835444209, 0.23149393417119404

In [ ]:
plt.hist(parameter_gradients, bins=50)
plt.xlabel("Gradient")
plt.ylabel("Count")
plt.show

In [ ]:
#speed testing CPU vs GPU

import time

start = time.perf_counter()

n_trials = 100

for trials in range(n_trials):

    weights = np.random.uniform(-np.pi,np.pi,size=(n_layers,n_qubits),requires_grad=True)

    gradient = qml.grad(circuit)(weights)

elapsed = time.perf_counter() - start

print(f"{n_trials} trials: {elapsed:.2f} seconds")

# CPU 100 Trials @ 1.23s
# GPU 100 Trials @ 0.64
